# Calculate Hargreaves potential Evapotranspiration for CAMELS-DE-1h

In [1]:
import pandas as pd
import numpy as np
import xarray as xr
from glob import glob
import pickle
from tqdm import tqdm
from joblib import Parallel, delayed

In [ ]:
camelsde1h_path = "../data/CAMELS-DE-1h"

df_topo = pd.read_csv(f"{camelsde1h_path}/CAMELS_DE_1h_topographic_attributes.csv")

ids = [id.split("_")[-1].split(".")[0] for id in glob(f"{camelsde1h_path}/timeseries/*.csv")]
len(ids)

In [3]:
def calculate_hourly_hargreaves_pet(df, lat, lon, coeff=0.0013):
    """
    Calculates hourly Potential Evapotranspiration (PET) using a modified 
    Hargreaves-Samani equation adapted for Central European climates.

    This version accounts for hourly solar geometry, longitude-based time 
    correction, and monthly precipitation/temperature range proxies.

    Parameters:
        df (pd.DataFrame): Hourly weather data. Index must be datetime (UTC+0).
            Required columns:
            - 'temperature_mean_mean': Hourly average temperature [°C].
            - 'precipitation_sum_mean': Hourly precipitation [mm].
        lat (float): Latitude in decimal degrees.
        lon (float): Longitude in decimal degrees (Positive East).
        coeff (float): Empirical coefficient. Default is 0.0013 (calibrated 
            for Germany/Central Europe). Use 0.0023 for arid/original 
            Hargreaves-Samani applications.

    Returns:
        pd.Series: Hourly PET values [mm/hour].

    Notes:
        - The function applies a longitude correction to align UTC+0 data 
          with Local Solar Time.
        - Includes an orographic/moisture correction term (t_diff and mon_prec) 
          often used in German hydrological modeling.
        - Ensure data is cleaned; negative moisture terms are clipped to zero.
    """
    # 1. Climate Proxies (calculated from the hourly series)
    t_daily_max = df['air_temperature_mean_mean'].resample('D').transform('max')
    t_daily_min = df['air_temperature_mean_mean'].resample('D').transform('min')
    
    # Monthly smoothing for the temperature range and precipitation
    t_diff = (t_daily_max - t_daily_min).resample('ME').transform('mean').reindex(df.index, method='ffill')
    mon_prec = df['precipitation_sum_mean'].resample('ME').transform('mean').reindex(df.index, method='ffill')

    # 2. Solar Geometry
    phi = np.radians(lat)
    doy = df.index.day_of_year
    hour = df.index.hour + 0.5  # Midpoint of the hour
    
    delta = 0.409 * np.sin(2 * np.pi / 365 * doy - 1.39)  # Declination
    dr = 1 + 0.033 * np.cos(2 * np.pi / 365 * doy)       # Inverse relative distance

    # 3. Solar Time Correction (UTC+0 to Solar Noon)
    # Equation of Time
    b = 2 * np.pi * (doy - 81) / 364
    Et = 0.165 * np.sin(2 * b) - 0.126 * np.cos(b) - 0.025 * np.sin(b)
    
    # Correcting for longitude (15° per hour)
    solar_hour = hour + Et + (lon / 15.0)
    omega = (np.pi / 12) * (solar_hour - 12)
    
    # 4. Sunset Hour Angle
    omega_s = np.arccos(np.clip(-np.tan(phi) * np.tan(delta), -1, 1))

    # 5. Extraterrestrial Radiation (Ra)
    is_daylight = (omega > -omega_s) & (omega < omega_s)
    
    # 15.392 MJ/m2/day scaled for hourly integration and use (pi / 24) instead of (12 / pi) to scale daily potential to hourly intensity
    ra_hourly = np.where(
        is_daylight,
        (np.pi / 24) * 15.392 * dr * (
            (np.sin(phi) * np.sin(delta)) + 
            (np.cos(phi) * np.cos(delta) * np.cos(omega))
        ),
        0
    )

    # 6. Final Calculation
    # Moisture term adapted for Central Europe
    moisture_term = np.maximum(t_diff - 0.0123 * mon_prec, 0)
    pet = coeff * ra_hourly * (df['air_temperature_mean_mean'] + 17) * (moisture_term ** 0.76)
    
    return pd.Series(np.maximum(pet, 0), index=df.index, name='pet')

Parallel execution

In [ ]:
def process_single_catchment(id, camelsde1h_path, df_topo):
    """Worker function for a single catchment ID."""
    try:
        # 1. Load data
        file_path = f"{camelsde1h_path}/timeseries/CAMELS_DE_1h_hydromet_timeseries_{id}.csv"
        df = pd.read_csv(file_path, parse_dates=["time"], index_col="time")

        # 2. Get Metadata
        # Use .loc with a mask for better performance than full filtering
        meta = df_topo.loc[df_topo["gauge_id"] == id].iloc[0]
        lat, lon = meta["gauge_lat"], meta["gauge_lon"]
        
        # 3. Calculate PET
        pet_series = calculate_hourly_hargreaves_pet(df, lat, lon)
        return id, pet_series

    except Exception as e:
        return id, f"Error: {e}"

# n_jobs=-1 uses all available CPU cores
raw_results = Parallel(n_jobs=-1)(
    delayed(process_single_catchment)(id, camelsde1h_path, df_topo) 
    for id in tqdm(ids, desc="Calculating PET")
)

# 1. Filter out errors and build the dictionary
pet_results = {}
for catchment_id, result in raw_results:
    if isinstance(result, pd.Series):
        pet_results[catchment_id] = result
    else:
        print(f"Skipping ID {catchment_id} due to error: {result}")

# 2. Sort the dictionary by gauge_id
pet_results = dict(sorted(pet_results.items()))

print(f"Done! Successfully processed {len(pet_results)} catchments.")

In [ ]:
# 1. Convert dictionary items to a list of Xarray DataArrays
# This is lightweight because it wraps the existing data
das = []
print("Converting dictionary to Xarray objects...")

for gauge_id, series in pet_results.items():
    # Create a DataArray for this specific series
    da = xr.DataArray(
        series.values,
        coords={"time": series.index},
        dims="time",
        name="pet_mm_h"
    )
    # Add the ID as a coordinate so we can stack them later
    da = da.assign_coords(gauge_id=gauge_id)
    das.append(da)

# 2. Combine into a single Dataset
# xr.concat aligns the time dimension automatically (inserting NaNs if ranges differ)
print("Concatenating (this might take a moment)...")
combined_da = xr.concat(das, dim="gauge_id")

# 3. Convert to Dataset
ds = combined_da.to_dataset()

# Strip the timezone using the underlying Pandas Index
ds = ds.assign_coords(time=ds.indexes["time"].tz_localize(None))

# Verify the change (Coordinates should now show just 'datetime64[ns]')
print("New time dtype:", ds.time.dtype)

# rename variable
ds = ds.rename({"pet_mm_h": "pet"})

# 4. Save to NetCDF
output_nc = "../data/CAMELS_DE_1h/pet_hargreaves.nc"
print(f"Saving NetCDF to {output_nc}...")

# Zlib compression is highly recommended for this size (3GB -> ~300MB)
encoding = {"pet": {"zlib": True, "complevel": 5}}
ds.to_netcdf(output_nc, encoding=encoding)

print("Done.")

print("NetCDF save successful.")